# 05 — Inference: real super-resolved output

Applies the trained model (`notebooks/04_train.ipynb`, `models/dsen2_20m.pt`) to a **real** Sentinel-2 scene — not the synthetic Wald's-protocol val split notebook 4 evaluated on. This is the actual deliverable: a proper, georeferenced 10-band GeoTIFF at native 10m resolution — the 4 native 10m bands (B02/B03/B04/B08, passed through unchanged) plus the 6 super-resolved 20m bands (B05/B06/B07/B8A/B11/B12) — usable directly in a downstream geoAI pipeline without a separate step to reassemble RGB.

Important limitation: there's no ground truth at 10m for the 20m bands in real data (that's the entire premise of the task), so unlike notebook 4 there's no PSNR/SAM here — only a qualitative visual check (does the output look sharper than bicubic and consistent with the real 10m guide's features, without obvious artifacts?), the same kind of eyeball check used throughout this project (see `agents.md`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.windows
import torch

sys.path.insert(0, str(Path.cwd().parent / "src"))

from s2sr import config, stac
from s2sr.dataset import upsample_bicubic
from s2sr.inference import superresolve_scene
from s2sr.model import DSen2Net20m
from s2sr.patches import patch_windows
from s2sr.raster import percentile_stretch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DSen2Net20m(guide_channels=len(config.BANDS_10M), target_channels=len(config.BANDS_20M)).to(device)
checkpoint_path = config.REPO_ROOT / "models" / "dsen2_20m.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Device: {device}")
print(f"Loaded checkpoint from epoch {checkpoint['epoch'] + 1}, val_loss={checkpoint['val_loss']:.4f}")

## Pick a scene

Unlike notebooks 2/3 (which re-search hundreds of scenes for live signed URLs), inference only needs one scene at a time, so `stac.fetch_item_by_id` fetches it directly — no need to re-run a full catalog search just to refresh one item's signed asset URLs.

In [ ]:
selected = gpd.read_file(config.REPO_ROOT / "data" / "interim" / "s2_scenes_ghana_2025_selected.geojson")
scene_id = selected.iloc[0]["id"]  # change the index to pick a different scene

item = stac.fetch_item_by_id(scene_id)
ghana_boundary = gpd.read_file(config.GHANA_BOUNDARY_PATH)
print(f"Scene: {item.id}")

In [ ]:
out_path = config.REPO_ROOT / "data" / "processed" / "superresolved" / f"{item.id}_20m_at_10m.tif"
superresolve_scene(item, ghana_boundary, model, device, out_path)
print(f"Saved: {out_path}")

## Visual check

One tile from the output, compared against the real 10m guide (for spatial context/detail) and the plain-bicubic baseline (the same comparison notebook 4 used, but now on real data with no synthetic degradation involved).

In [ ]:
with rasterio.open(out_path) as dst:
    aoi_geom = ghana_boundary.to_crs(dst.crs).union_all()
    window = next(iter(patch_windows(dst.width, dst.height, dst.transform, aoi_geom, config.PATCH_SIZE_10M)))

    rgb_band_idx = [config.BANDS_10M.index(b) + 1 for b in ("B04", "B03", "B02")]
    guide_rgb = percentile_stretch(np.stack([dst.read(i, window=window) for i in rgb_band_idx], axis=-1))

    sr_band_idx = len(config.BANDS_10M) + config.BANDS_20M.index("B11") + 1
    superresolved = dst.read(sr_band_idx, window=window)

native_20m_window = rasterio.windows.Window(
    window.col_off // config.DOWNSAMPLE_FACTOR_20M,
    window.row_off // config.DOWNSAMPLE_FACTOR_20M,
    config.PATCH_SIZE_10M // config.DOWNSAMPLE_FACTOR_20M,
    config.PATCH_SIZE_10M // config.DOWNSAMPLE_FACTOR_20M,
)
with rasterio.open(item.assets["B11"].href) as src:
    native_20m = src.read(1, window=native_20m_window).astype("float32")
bicubic_baseline = upsample_bicubic(native_20m[None], out_size=config.PATCH_SIZE_10M)[0]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(guide_rgb)
axes[0].set_title("guide (10m RGB)")
axes[1].imshow(bicubic_baseline, cmap="gray")
axes[1].set_title("bicubic baseline (B11)")
axes[2].imshow(superresolved, cmap="gray")
axes[2].set_title("model output (B11)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()

## Next steps

- Run this over more scenes (loop over `selected` instead of just `selected.iloc[0]`) once the single-scene output looks right.
- The output GeoTIFF only covers `patch_size`-aligned tiles fully inside the raster extent — same edge behavior as patch extraction (see agents.md). A production version would want edge handling (overlap-and-blend, or padding) if full scene coverage matters.
- Land-cover stratification (see agents.md open caveat / notebook 3) may improve robustness on land-cover types underrepresented in training.